In [1]:
import pandas as pd

In [10]:
import pandas as pd
import numpy as np
from scipy.stats import entropy

def calculate_column_entropy(column, base=2):
    # Get the counts of unique values in the column
    value_counts = column.value_counts()
    # Pass the counts to scipy.stats.entropy
    # The function normalizes the counts into probabilities internally
    return entropy(value_counts, base=base)

# Example DataFrame
data = {
    'Category': ['A', 'B', 'A', 'C', 'B', 'A', 'C', 'C'],
    'Value': [1, 2, 1, 3, 2, 1, 3, 3],
    'Binary': [0, 1, 0, 1, 0, 0, 1, 1]
}
df = pd.DataFrame(data)

# Calculate entropy for each column
def calculate_all_columns_entropy(data_frame):
    entropy_results = {}
    for col in data_frame.columns:
        entropy_results[col] = calculate_column_entropy(data_frame[col])
    return entropy_results

print(calculate_all_columns_entropy(df))


{'Category': np.float64(1.561278124459133), 'Value': np.float64(1.561278124459133), 'Binary': np.float64(1.0)}


In [9]:
# Calculate entropy for each column
def calculate_all_columns_entropy(data_frame):
    entropy_results = {}
    for col in data_frame.columns:
        entropy_results[col] = calculate_column_entropy(data_frame[col])
    return entropy_results

print(calculate_all_columns_entropy(data))

AttributeError: 'dict' object has no attribute 'columns'

In [3]:
df = pd.read_csv('/home/saad-naeem/Desktop/Level_4/Graduation_project_saad/chatbot/Baytology/egypt_real_estate_preprocessed.csv')

In [4]:
entropy_results = {}
for col in df.columns:
    entropy_results[col] = calculate_column_entropy(df[col])

print(entropy_results)

{'price': np.float64(9.535671591702584), 'location': np.float64(1.1718323174859537), 'type': np.float64(2.390111112686735), 'bedrooms': np.float64(2.1106350203852706), 'bathrooms': np.float64(2.318544158712618), 'payment_method': np.float64(0.7204190598003256), 'size_sqm': np.float64(7.906146094768993)}


In [2]:
# The original dictionary
data = {'apple': 5, 'banana': 2, 'cherry': 8, 'date': 0}

# Sort by value (item[1]) in ascending order
sorted_items = sorted(data.items(), key=lambda item: item[1])

# Convert the sorted list of tuples back into a dictionary
sorted_data = dict(sorted_items)

print(sorted_data)
# Output: {'date': 1, 'banana': 2, 'apple': 5, 'cherry': 8}


{'date': 0, 'banana': 2, 'apple': 5, 'cherry': 8}


In [3]:
from parser.parse_user_txt import parse_user_query
from parser.data_validation import RealEstateQuery
from search_engine.search_engine import load_data, filter_properties
from model.helper_functions import score_and_rank, calculate_all_columns_entropy
from langchain_google_genai import ChatGoogleGenerativeAI
import os
import joblib
from dotenv import load_dotenv  




# change the current working dir
os.chdir(os.getcwd())

# Load the .env file
load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

# get the gemini api key
os.environ["GOOGLE_API_KEY"] = api_key

# 1. Load Data
df = load_data("../Baytology/egypt_real_estate_preprocessed.csv")

# 2. Get User Input
user_text = "عايز شقه في القاهرة ب 3 مليون و فيها 3 خمامات"


# 3. CONFIGURE GEMINI
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0  # 0 means "be precise, don't be creative"
)

structured_llm = llm.with_structured_output(RealEstateQuery)


# 3. Phase 1: AI Understanding
print("🤖 AI is thinking...")

# parse the user input
filters = parse_user_query(structured_llm=structured_llm,text=user_text)

print(f"\n🔍 Extracted Filters: {filters}")


# 4. Phase 2: Database Search
matches = filter_properties(df, filters)
# print(type(matches))
print(" ")
print(calculate_all_columns_entropy(matches))

# print(f"\n🏠 Found {len(matches)} houses matching your request.")

# if len(matches) > 0:
#     print(matches.head())
# else:
#     print("No houses found! Try increasing your budget.")


# ==========================================
# 6. PHASE 3: AI JUDGE (The Ranker)
# ==========================================
# 1. LOAD THE SAVED BRAINS


# We load the model and the translators we just created
try:
    model = joblib.load('../Baytology/model/price_model.pkl')
    le_location = joblib.load('../Baytology/model/location_encoder.pkl')
    le_type = joblib.load('../Baytology/model/type_encoder.pkl')
    le_payment = joblib.load('../Baytology/model/payment_encoder.pkl')
    # print("\n✅ AI Judge loaded successfully.")

except FileNotFoundError:
    print("❌ Error: Model files not found. Run train_model.py first.")



if not matches.empty:
    print("\n⭐ AI Judge is evaluating deals (Value for Money)...")
    
    # 👇 CALL THE RANKER HERE
    # This adds 'fair_price' and 'deal_score' columns and sorts them
    best_houses = score_and_rank(candidates_df=matches,le_location=le_location, le_type=le_type,le_payment=le_payment,model=model)
    
    # Show Top 3 Recommendations
    print(f"TOP 5 RECOMMENDED DEALS:")
    for index, row in best_houses.head(5).iterrows():
        print("-" * 30)
        print(f"{row['type']} in {row['location']}")
        print(f"Price: {row['price']:,.0f} EGP")
        print(f"Fair Value: {row['fair_price']:,.0f} EGP")

        print(f"bedrooms: {row['bedrooms']}")
        print(f"bathrooms: {row['bathrooms']}")
        print(f"payment_method: {row['payment_method']}")
        print(f"size: {row['size_sqm']}")
        
        # Explain the score
        score = row['deal_score']
        status = "Good Deal (Undervalued)" if score > 0 else "Premium/Overpriced"
        print(f"Deal Score: {score:,.0f} ({status})")
        
else:
    print("No houses found! Try increasing your budget or changing location.")

ModuleNotFoundError: No module named 'parser'